# TPV104 — rate-and-state friction with strong velocity weakening

TPV104 is a benchmark exercise of the [SCEC/USGS Spontaneous Rupture Code Verification Project](https://strike.scec.org/cvws/) ([Harris et al., SRL 2018](https://pubs.geoscienceworld.org/ssa/srl/article/89/3/1146/530061/A-Suite-of-Exercises-for-Verifying-Dynamic)). It features:

* spontaneous rupture on a **vertical, planar strike-slip fault** embedded in a homogeneous half-space ($V_p=6000\,m/s,\ V_s=3464\,m/s,\ \rho=2670\,kg/m^3$)
* **rate-and-state friction, slip law, with strong velocity weakening** (SeisSol friction law `FL=103`): the friction coefficient drops sharply toward $f_w=0.2$ once the slip rate exceeds the weakening rate $V_w$
* a central **velocity-weakening (VW)** core ($|x|\le 15$ km, depth $\le 15$ km) surrounded by a **velocity-strengthening (VS)** border, with a smooth 3 km `tanh` transition (SCEC Eq. 4-5)
* a **forced nucleation** patch: a compact-bell shear overstress (up to 45 MPa) within 3 km of the hypocentre at $(0, 0, -7.5\,\text{km})$, ramped in over $t_0=1$ s

This is the companion of the linear-slip-weakening TPV13 example, using the same SeisSol App on the Quakeworx Gateway. See the [detailed benchmark description (SCEC)](https://strike.scec.org/cvws/tpv104docs.html).

### Input files in this folder

| File | Role |
|------|------|
| `parameters.par` | SeisSol main parameter file (`FL=103`) |
| `tpv104_fault.yaml` | fault friction (spatial `rs_a`, `rs_srW`), initial stress, and nucleation overstress |
| `tpv104_material.yaml` | homogeneous elastic material ($\rho,\mu,\lambda$) |
| `tpv104_faultreceivers.dat` | 9 SCEC on-fault stations |
| `tpv104_receivers.dat` | 6 off-fault (free-surface) receivers |
| `tpv104.puml.h5` | the SeisSol PUML mesh (see *Mesh* below) |

## Mesh

SeisSol reads the mesh in the efficient HDF5-based **PUML** format (`tpv104.puml.h5`). Here we **reuse the full 200 m TPV104 tetrahedral mesh** generated for the MFEM port (`miniapps/seas/tpv104/mesh/tpv104_200m.msh`): a vertical strike-slip fault at `y=0`, sliding region $x\in[-18,18]$ km, depth $z\in[-18,0]$ km, refined to ~200 m near the fault (the SCEC process zone $\Lambda_\mathrm{dyn}\approx 160$ m is resolved) and coarsening outward.

That `.msh` already carries SeisSol's boundary-condition tags directly — `Physical Surface 1 = free surface, 3 = dynamic rupture, 5 = absorbing` — the same convention as the tpv13 training mesh, so **no retagging is needed**.

The final `.msh` -> `.puml.h5` conversion uses [PUMGen](https://github.com/SeisSol/PUMGen), which only compiles on Linux. On a Linux machine (e.g. a login node on the cluster behind Quakeworx) run the helper script in this folder (it builds PUMGen and runs the conversion):

```bash
bash build_puml_mesh.sh        # tpv104.msh -> tpv104.puml.h5
```

or, if `pumgen` is already on your PATH, simply `pumgen -s msh2 tpv104.msh`. See `MESH_README.md` for full provenance and the verified BC-code convention.

In [ ]:
# Build the PUML mesh (Linux only — PUMGen does not build on macOS arm64).
# Uncomment to run on a Linux machine that has this folder:
# !bash build_puml_mesh.sh

## Run SeisSol on the Quakeworx Gateway

We run SeisSol using the provided `parameters.par`.

**Please go to the [Quakeworx Gateway](https://qwx1.onescienceway.com) now**, select the SeisSol App, upload **all six** input files below, and submit the job.

> **Upload checklist — all of these must be in the run directory.** The run aborts at startup (`cannot open file: ...`) if *any* file is missing. The two small `.dat` receiver files are easy to forget:
>
> - [ ] `parameters.par`
> - [ ] `tpv104_material.yaml`
> - [ ] `tpv104_fault.yaml`
> - [ ] `tpv104_faultreceivers.dat`  ← on-fault stations (required by `OutputPointType = 5`)
> - [ ] `tpv104_receivers.dat`  ← off-fault receivers (required by `RFileName`)
> - [ ] `tpv104.puml.h5`  ← the mesh (see *Mesh* above)

While SeisSol is running, you might want to check out the [documentation of the input files](https://seissol.readthedocs.io/en/latest/parameter-file.html).

## Fault frictional structure (VW core vs VS border)

Before looking at the results, we visualize the spatial friction parameters that `tpv104_fault.yaml` defines on the fault. The direct-effect parameter $a$ and the weakening rate $V_w$ are interpolated between the velocity-weakening core and the velocity-strengthening border by the SCEC smooth boxcar $B(x,z)$ (the same `tanh` formula used in the yaml):

$$a = 0.01 + 0.01\,(1 - B),\qquad V_w = 0.1 + 0.9\,(1 - B)\ \text{m/s},$$

so $a=0.01,\ V_w=0.1$ (weakening) in the core and $a=0.02,\ V_w=1.0$ (strengthening) outside.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def boxcar(x, z):
    # exact reproduction of the tanh boxcar in tpv104_fault.yaml
    x1 = np.abs(x)
    bx = 0.5 * (1.0 + np.tanh(3000.0/(x1 - 18000.0) + 3000.0/(x1 - 15000.0)))
    bx = np.where(x1 >= 18000.0, 0.0, bx)
    bx = np.where(x1 <= 15000.0, 1.0, bx)
    z1 = np.abs(z + 7500.0)
    bz = 0.5 * (1.0 + np.tanh(3000.0/(z1 - 10500.0) + 3000.0/(z1 - 7500.0)))
    bz = np.where(z1 >= 10500.0, 0.0, bz)
    bz = np.where(z1 <= 7500.0, 1.0, bz)
    return bx * bz

xs = np.linspace(-20e3, 20e3, 401)
zs = np.linspace(-20e3, 0.0, 201)
X, Z = np.meshgrid(xs, zs)
B = boxcar(X, Z)
a   = 0.01 + 0.01 * (1.0 - B)
V_w = 0.1  + 0.9  * (1.0 - B)

fig, ax = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for axi, field, label in zip(ax, (a, V_w), ('direct-effect a', 'weakening rate V_w [m/s]')):
    pc = axi.pcolormesh(X/1e3, Z/1e3, field, shading='auto', cmap='viridis')
    axi.add_patch(plt.Circle((0, -7.5), 3.0, fill=False, ec='r', lw=1.5, ls='--'))  # nucleation
    axi.set_xlabel('along-strike x [km]'); axi.set_ylabel('depth z [km]')
    axi.set_title(label); axi.set_aspect('equal')
    fig.colorbar(pc, ax=axi, shrink=0.85)
ax[0].text(0, -7.5, 'nucleation', color='r', ha='center', va='center', fontsize=8)
plt.show()

## Visualization of the SeisSol output

We now visualize the fault output generated by SeisSol. Check the [fault-output documentation](https://seissol.readthedocs.io/en/latest/fault-output.html#outputmask) for the meaning of each field abbreviation (`SRs` = along-strike slip rate, `SRd` = down-dip slip rate, `Vr` = rupture speed, `ASl` = accumulated slip, ...).

In [ ]:
!pip install seissolxdmf
import sys
sys.path.append('/home/qwxdev/.local/lib/python3.12/site-packages')

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import seissolxdmf as seisx

### Set the path to your simulation output

This depends on the job name you chose in the Gateway. **Please use your job name here.**

In [ ]:
# Assuming your job name was 'SeisSol_TPV104'
path = 'SeisSol_TPV104'

### Slip rate and rupture speed on the fault

The TPV104 fault is the vertical plane `y = 0`, so we plot the fault fields in the along-strike ($x$) - depth ($z$) plane. Pick a time step `ndt` to inspect.

In [ ]:
xdmfFilename = '../../../{}/outputs/tpv104-fault.xdmf'.format(path)
sx = seisx.seissolxdmf(xdmfFilename)
ndt_max = sx.ReadNdt() - 1
print('number of fault output snapshots:', ndt_max + 1)

ndt = min(8, ndt_max)              # choose a snapshot during rupture
xyz = sx.ReadGeometry()            # (nnode, 3) triangle vertices
connect = sx.ReadConnect()         # (ntri, 3) connectivity
SRs = sx.ReadData('SRs', ndt)      # along-strike slip rate (per triangle)
Vr  = sx.ReadData('Vr',  ndt)      # rupture speed (per triangle)

x = xyz[:, 0]; z = xyz[:, 2]       # fault is at y ~ 0; plot x vs z

fig, ax = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
tp0 = ax[0].tripcolor(x/1e3, z/1e3, connect, facecolors=SRs, cmap='magma')
ax[0].set_title('along-strike slip rate SRs [m/s]')
fig.colorbar(tp0, ax=ax[0], shrink=0.85)
tp1 = ax[1].tripcolor(x/1e3, z/1e3, connect, facecolors=np.clip(Vr, 0, 6000), cmap='viridis')
ax[1].set_title('rupture speed Vr [m/s]')
fig.colorbar(tp1, ax=ax[1], shrink=0.85)
for axi in ax:
    axi.set_xlabel('along-strike x [km]'); axi.set_ylabel('depth z [km]')
    axi.set_aspect('equal'); axi.set_xlim(-20, 20); axi.set_ylim(-20, 0)
plt.show()

### On-fault station time series (the 9 SCEC TPV104 stations)

SeisSol writes one ASCII time series per on-fault receiver listed in `tpv104_faultreceivers.dat` (in Tecplot style, with a `VARIABLES = ...` header naming the columns). The loader below parses that header so it works regardless of the exact `OutputMask` column order. The 9 stations correspond to the standard SCEC TPV104 on-fault locations (named here `x2 = along-strike`, `x3 = depth`, both in km).

In [ ]:
import glob, os

# Station coordinates (x, y, z) in tpv104_faultreceivers.dat order, with SCEC labels.
stations = [
    ((0e3,    0, -3e3),   'x2=0 x3=3'),
    ((0e3,    0, -7.5e3), 'x2=0 x3=7.5'),
    ((0e3,    0, -12e3),  'x2=0 x3=12'),
    ((9e3,    0, -7.5e3), 'x2=9 x3=7.5'),
    ((12e3,   0, -3e3),   'x2=12 x3=3'),
    ((12e3,   0, -12e3),  'x2=12 x3=12'),
    ((-9e3,   0, -7.5e3), 'x2=-9 x3=7.5'),
    ((-12e3,  0, -3e3),   'x2=-12 x3=3'),
    ((-12e3,  0, -12e3),  'x2=-12 x3=12'),
]

def load_receiver(fname):
    """Parse a SeisSol fault-receiver .dat (Tecplot style). Returns (col_names, data)."""
    names, data = None, []
    with open(fname) as fh:
        for line in fh:
            s = line.strip()
            if s.upper().startswith('VARIABLES'):
                rhs = s.split('=', 1)[1]
                names = [t.strip().strip('"') for t in rhs.split(',')]
            elif s and not s[0].isalpha() and not s.startswith('#'):
                try:
                    data.append([float(v) for v in s.split()])
                except ValueError:
                    pass
    return names, np.array(data)

recdir = '../../../{}/outputs'.format(path)
files = sorted(glob.glob(os.path.join(recdir, 'tpv104*faultreceiver*.dat')))
print('found {} receiver files in {}'.format(len(files), recdir))

fig, axes = plt.subplots(3, 3, figsize=(15, 10), sharex=True, constrained_layout=True)
for k, (ax, (coord, label)) in enumerate(zip(axes.ravel(), stations)):
    if k < len(files):
        names, dat = load_receiver(files[k])
        if dat.size and names:
            tcol = names.index('Time') if 'Time' in names else 0
            # plot along-strike slip rate SRs if present, else first slip-rate column
            scol = names.index('SRs') if 'SRs' in names else 1
            ax.plot(dat[:, tcol], dat[:, scol], lw=1.2)
            ax.set_ylabel('SRs [m/s]')
    ax.set_title(label); ax.grid(alpha=0.3)
for ax in axes[-1]:
    ax.set_xlabel('time [s]')
fig.suptitle('TPV104 on-fault slip rate (SRs) time series', y=1.02)
plt.show()

### Coseismic ground deformation (free surface)

Finally we visualize the vertical and horizontal surface displacement produced by the rupture (SeisSol `SurfaceOutput`).

In [ ]:
def plot_surface(xdmfFilename, timestep, comp='u1'):
    sx = seisx.seissolxdmf(xdmfFilename)
    xyz = sx.ReadGeometry()
    connect = sx.ReadConnect()
    U = sx.ReadData(comp, timestep)            # u1,u2 = horizontal; u3 = vertical
    print('{}: min={:.3f}  max={:.3f} m'.format(comp, float(np.min(U)), float(np.max(U))))
    x, y = xyz[:, 0]/1e3, xyz[:, 1]/1e3
    vmax = max(1e-6, np.nanmax(np.abs(U)))
    plt.tripcolor(x, y, connect, facecolors=U, cmap='seismic', vmin=-vmax, vmax=vmax)
    plt.xlabel('Easting x [km]'); plt.ylabel('Northing y [km]'); plt.gca().set_aspect('equal')
    plt.colorbar(orientation='horizontal', label='{} displacement [m]'.format(comp))

xdmfFilename = '../../../{}/outputs/tpv104-surface.xdmf'.format(path)
sxs = seisx.seissolxdmf(xdmfFilename)
tlast = sxs.ReadNdt() - 1
plt.figure(figsize=(9, 6))
plot_surface(xdmfFilename, tlast, comp='u1')   # fault-parallel horizontal displacement
plt.show()

### Exercises

* Change the fault field plotted (`SRs`, `SRd`, `ASl`, `Vr`, ...) and the snapshot index `ndt`.
* Compare the slip-rate time series at the central station (`x2=0 x3=7.5`) against the published SCEC TPV104 results.
* In `tpv104_fault.yaml`, widen or narrow the VW core (the `15000`/`18000` half-widths) and observe how the rupture extent changes.
* Re-run with rate-2 local time-stepping (`ClusteredLTS = 2`) vs global time-stepping (`ClusteredLTS = 1`) and compare the wall-clock time in the SeisSol log.